# 🌊 Flood Segmentation — ANRFAISEHack Theme 1 Phase 2

> **Task:** Pixel-wise segmentation of multi-band SAR satellite imagery to detect flooded regions using a deep learning encoder–decoder network.

---

## 🗺️ Pipeline Overview

```
6-channel SAR GeoTIFF
       │
       ▼
 FloodDataset (PyTorch Dataset)
       │
       ▼
 UNet  (ResNet-18 encoder, ImageNet weights)
       │
       ▼
 3-class logits  ──►  CrossEntropyLoss  ──►  Adam optimiser
       │
       ▼
 argmax prediction  ──►  binary flood mask  ──►  RLE  ──►  submission.csv
```

| Hyperparameter | Value |
|---|---|
| Architecture | UNet |
| Encoder | ResNet-18 (ImageNet pretrained) |
| Input channels | 6 (SAR bands) |
| Output classes | 3 |
| Loss function | CrossEntropyLoss |
| Optimiser | Adam — lr 1e-4 |
| Epochs | 10 |
| Train batch size | 2 |
| Test batch size | 1 |

---
## 📦 Cell 1 — Install Dependencies

Three things are handled here:

| Step | Why |
|---|---|
| Reinstall PyTorch (cu121) | Fixes `AcceleratorError: no kernel image` — happens when Kaggle's default PyTorch was compiled for a different CUDA architecture than the assigned GPU |
| Install `segmentation-models-pytorch` | Ready-made UNet / FPN / DeepLab architectures with pretrained encoders |
| Install `imagecodecs` | Codec library used by `tifffile` to decompress LZW, JPEG-2000, ZSTD TIFF variants |

> ⚡ The `-q` flag suppresses verbose pip output. The `--index-url` points to PyTorch's official CUDA 12.1 wheel index, which is compatible with all current Kaggle GPU types (T4, P100, V100).

In [1]:
import os
import sys
import subprocess

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

print("🔄 Uninstalling PyTorch and related packages...")
# Uninstall PyTorch and related packages
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                       "torch", "torchvision", "torchaudio", "pytorch", 
                       "segmentation-models-pytorch", "pytorch-lightning"])

print("✅ Uninstall complete!")
print("\n📦 Installing Kaggle-supported PyTorch with CUDA 12.1...")

# Install Kaggle-supported PyTorch (CUDA 12.1 - compatible with all Kaggle GPUs: T4, P100, V100, A100)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                       "torch", "torchvision", "torchaudio",
                       "--index-url", "https://download.pytorch.org/whl/cu121"])

print("✅ PyTorch installed!")
print("\n📦 Installing segmentation libraries...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "segmentation-models-pytorch", "imagecodecs"])

print("\n✅ All dependencies installed successfully!")

# Verify installation
import torch
print(f"\nPyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("✅ Ready to train on GPU!")

🔄 Uninstalling PyTorch and related packages...


✅ Uninstall complete!

📦 Installing Kaggle-supported PyTorch with CUDA 12.1...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 97.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 74.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 43.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 101.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 15.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 35.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 15.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

---
## 🔬 Cell 2 — Imports & Device Setup

Standard scientific-Python stack plus the two domain-specific libraries:

| Library | Role |
|---|---|
| `numpy` | Array math and mask manipulation |
| `pandas` | Building and saving the submission CSV |
| `torch` / `torch.nn` | Model definition, training loop, loss |
| `tifffile` | Reading multi-band GeoTIFF rasters |
| `segmentation_models_pytorch` | UNet with pretrained ResNet-18 encoder |
| `tqdm` | Progress bars for training and inference |

**Device selection:** The code automatically picks **CUDA** (Kaggle GPU) if available, otherwise falls back to **CPU**.
Always enable the GPU accelerator in *Settings → Accelerator* for this notebook for best performance.

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tifffile
import segmentation_models_pytorch as smp
from tqdm import tqdm

# ── Device ──────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {device}")
if device.type == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print("⚠️  If you see CUDA errors during training, the code will automatically use CPU.")

# Flag to track if we need to fall back to CPU during training
_cuda_fallback = False

PyTorch  : 2.5.1+cu121
Device   : cuda
GPU      : Tesla P100-PCIE-16GB
⚠️  If you see CUDA errors during training, the code will automatically use CPU.


---
## 📁 Cell 3 — Dataset Paths

All paths are rooted at the competition's Kaggle input directory.

```
data/
├── image/               ← training SAR images   (*_image.tif)
├── label/               ← training masks        (*_label.tif)
├── prediction/
│   └── image/           ← test SAR images       (*_image.tif)
└── split/
    └── train.txt        ← newline-separated list of train IDs
```

> 💡 If you fork this notebook or change the competition slug, update only `DATA_PATH` — all other paths derive from it automatically.

A quick directory check is printed so you know immediately if any path is misconfigured.

In [3]:
DATA_PATH = "/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data"
IMG_DIR   = f"{DATA_PATH}/image"            # training images
LBL_DIR   = f"{DATA_PATH}/label"            # training masks
PRED_DIR  = f"{DATA_PATH}/prediction/image" # test images

# Quick sanity-check — confirm all directories exist
for name, path in [("IMG_DIR", IMG_DIR), ("LBL_DIR", LBL_DIR), ("PRED_DIR", PRED_DIR)]:
    status = "✅" if os.path.isdir(path) else "❌ NOT FOUND"
    print(f"{name:10s}: {status}  ({path})")

IMG_DIR   : ✅  (/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/image)
LBL_DIR   : ✅  (/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/label)
PRED_DIR  : ✅  (/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/prediction/image)


---
## 🗂️ Cell 4 — Load Sample IDs

- **Train IDs** are read from `split/train.txt` — one bare ID per line (no file extension).
- **Test IDs** are discovered by scanning `PRED_DIR` for `*_image.tif` files and stripping the suffix. They are sorted for deterministic ordering.

A small preview of both ID lists is printed so you can verify that files are being found correctly.

In [4]:
def load_ids(path):
    """Read a plain-text list of sample IDs (one per line, blank lines ignored)."""
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]

train_ids = load_ids(f"{DATA_PATH}/split/train.txt")

test_ids = sorted([
    f.replace("_image.tif", "")
    for f in os.listdir(PRED_DIR)
    if f.endswith("_image.tif")
])

print(f"Train samples : {len(train_ids):,}")
print(f"Test  samples : {len(test_ids):,}")
print(f"\nFirst 3 train IDs : {train_ids[:3]}")
print(f"First 3 test  IDs : {test_ids[:3]}")

Train samples : 59
Test  samples : 19

First 3 train IDs : ['20240529_EO4_RES2_fl_pid_001', '20240529_EO4_RES2_fl_pid_002', '20240529_EO4_RES2_fl_pid_003']
First 3 test  IDs : ['20240529_EO4_RES2_fl_pid_080', '20240529_EO4_RES2_fl_pid_081', '20240529_EO4_RES2_fl_pid_082']


---
## 🛢️ Cell 5 — Custom Dataset (`FloodDataset`)

`FloodDataset` is a standard PyTorch `Dataset` that handles both the **training** and **inference** splits through a single `train` flag.

### What happens inside `__getitem__`

| Step | Detail |
|---|---|
| 1. Load image | `tifffile` opens the 6-band SAR GeoTIFF → `(H, W, 6)` float32 array |
| 2. Transpose | `(H, W, 6)` → `(6, H, W)` to match PyTorch's channels-first convention |
| 3. Load mask *(train)* | Label TIFF read as `int64` — required by `CrossEntropyLoss` |
| 4. Return | Train → `(image_tensor, mask_tensor)` · Inference → `(image_tensor, id_string)` |

A smoke-test at the bottom of the cell loads one sample and prints its shape and unique class values so you can verify the dataset is wired up correctly.

In [5]:
class FloodDataset(Dataset):
    """PyTorch Dataset for 6-channel SAR flood segmentation."""

    def __init__(self, ids, train=True):
        self.ids   = ids
        self.train = train

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        import imagecodecs  # noqa: F401 — activates tifffile codec support
        id_ = self.ids[idx]

        # ── Image ────────────────────────────────────────────────────────────
        img_dir  = IMG_DIR if self.train else PRED_DIR
        img_path = os.path.join(img_dir, id_ + "_image.tif")
        with tifffile.TiffFile(img_path) as tif:
            img = tif.asarray().astype(np.float32)  # (H, W, 6)
        img = np.transpose(img, (2, 0, 1))           # → (6, H, W)

        # ── Mask (training only) ─────────────────────────────────────────────
        if self.train:
            mask_path = os.path.join(LBL_DIR, id_ + "_label.tif")
            with tifffile.TiffFile(mask_path) as tif:
                mask = tif.asarray().astype(np.int64)  # (H, W) class indices
            return torch.tensor(img), torch.tensor(mask)

        return torch.tensor(img), id_

# ── Smoke-test: load one training sample ────────────────────────────────────
sample_img, sample_mask = FloodDataset(train_ids, train=True)[0]
print(f"Image shape  : {tuple(sample_img.shape)}   dtype: {sample_img.dtype}")
print(f"Mask  shape  : {tuple(sample_mask.shape)}  dtype: {sample_mask.dtype}")
print(f"Mask classes : {sample_mask.unique().tolist()}")

Image shape  : (6, 512, 512)   dtype: torch.float32
Mask  shape  : (512, 512)  dtype: torch.int64
Mask classes : [0, 1, 2]


---
## 🔄 Cell 6 — DataLoaders

| Setting | Train | Test | Reason |
|---|---|---|---|
| `batch_size` | 2 | 1 | Small train batch fits in GPU memory; test runs one-by-one for clean ID tracking |
| `shuffle` | ✅ | ❌ | Randomises training order each epoch; test order must stay deterministic |
| `num_workers` | 0 | 0 | Avoids fork issues with `tifffile`/`imagecodecs` on Kaggle |
| `pin_memory` | auto | auto | Enabled on GPU for faster CPU→GPU data transfer |

> 💡 If you have a large dataset and want faster loading, try `num_workers=2`. It is safe on most Kaggle environments.

In [6]:
train_loader = DataLoader(
    FloodDataset(train_ids, train=True),
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)
test_loader = DataLoader(
    FloodDataset(test_ids, train=False),
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

print(f"Train loader : {len(train_loader):,} batches")
print(f"Test  loader : {len(test_loader):,} batches")

Train loader : 30 batches
Test  loader : 19 batches


---
## 🧠 Cell 7 — Model Definition (UNet + ResNet-18)

We use **SMP's UNet** with a **ResNet-18** encoder pretrained on ImageNet.

### Architecture choices

| Parameter | Value | Reason |
|---|---|---|
| `encoder_name` | `resnet18` | Lightweight and fast — good baseline for competition prototyping |
| `encoder_weights` | `imagenet` | Transfer learning speeds convergence even on non-RGB inputs |
| `in_channels` | `6` | SAR imagery has 6 spectral bands |
| `classes` | `3` | Background / Non-flood / Flood |

SMP automatically adapts the first convolution layer to accept 6 input channels while keeping the remaining pretrained weights intact.

> 🔧 **Want to experiment?** Try swapping `encoder_name` to `"resnet34"`, `"efficientnet-b2"`, or `"mit_b2"` with no other changes needed.

In [7]:
model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=6,
    classes=3,
).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print("Model ready ✅")

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Total parameters     : 14,337,907
Trainable parameters : 14,337,907
Model ready ✅


---
## ⚙️ Cell 8 — Loss Function & Optimiser

### Loss — `CrossEntropyLoss`
The standard loss for multi-class segmentation. It combines `LogSoftmax + NLLLoss` internally, so the model outputs raw logits (no softmax needed before the loss).

### Optimiser — `Adam`
Adam is a solid default for segmentation tasks.

| Setting | Value |
|---|---|
| Learning rate | `1e-4` |
| β₁ | 0.9 (default) |
| β₂ | 0.999 (default) |
| Weight decay | 0 (default) |

> 💡 **Optional improvement:** Add `torch.optim.lr_scheduler.CosineAnnealingLR` for a gradual learning-rate warm-down across the 10 epochs.

In [8]:
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print(f"Loss      : {loss_fn.__class__.__name__}")
print(f"Optimiser : {optimizer.__class__.__name__}  (lr={optimizer.param_groups[0]['lr']})")
print("Loss & optimiser ready ✅")

# Load best model for inference (if it exists)
import os
if os.path.exists('best_model.pth'):
    model.load_state_dict(torch.load('best_model.pth', map_location=device))
    print("✅ Loaded best model for inference")
else:
    print("ℹ️  Using final trained model (no best_model.pth found)")

Loss      : CrossEntropyLoss
Optimiser : Adam  (lr=0.0001)
Loss & optimiser ready ✅
ℹ️  Using final trained model (no best_model.pth found)


---
## 🏋️ Cell 9 — Training Loop

### Per-epoch steps

```
for each batch:
    1. Zero gradients          (optimizer.zero_grad)
    2. Forward pass            (model → logits shape: B×3×H×W)
    3. Compute loss            (CrossEntropyLoss vs ground-truth mask)
    4. Backward pass           (loss.backward — compute gradients)
    5. Update weights          (optimizer.step)
    6. Accumulate batch loss
→ Print average epoch loss + track best epoch
```

A `history` list records the average loss per epoch. The best epoch is printed at the end.

> ⏱️ **Expected runtime:** ~2–5 min per epoch on a Kaggle T4 GPU depending on dataset size.

In [9]:
EPOCHS  = 10
history = []  # per-epoch average loss
best_loss = float('inf')
best_epoch = 0

try:
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0

        for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1:>2}/{EPOCHS}", leave=False):
            imgs, masks = imgs.to(device), masks.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)           # (B, 3, H, W) raw logits
            loss    = loss_fn(outputs, masks)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg = total_loss / len(train_loader)
        history.append(avg)

        # Save best model
        if avg < best_loss:
            best_loss = avg
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'best_model.pth')
            print(f"Epoch {epoch+1:>2}/{EPOCHS}  |  avg loss: {avg:.4f}  ⭐ BEST")
        else:
            print(f"Epoch {epoch+1:>2}/{EPOCHS}  |  avg loss: {avg:.4f}")

    print("\nTraining complete  ✅")
    print(f"Best epoch loss : {best_loss:.4f}  (epoch {best_epoch})")
    print("Best model saved as 'best_model.pth'")

except RuntimeError as e:
    if "no kernel image" in str(e) or "CUDA" in str(e):
        print("\n⚠️  CUDA compatibility error detected! Switching to CPU...")
        print(f"   Error: {str(e)[:100]}...\n")

        # Move model and data to CPU
        device = torch.device("cpu")
        model = model.cpu()
        loss_fn = loss_fn.cpu() if hasattr(loss_fn, 'cpu') else loss_fn

        # Recreate dataloaders without CUDA
        train_loader = DataLoader(
            FloodDataset(train_ids, train=True),
            batch_size=2,
            shuffle=True,
            num_workers=0,
            pin_memory=False,
        )

        # Restart training on CPU (slower but compatible with P100)
        print("Restarting training on CPU...\n")
        for epoch in range(EPOCHS):
            model.train()
            total_loss = 0.0

            for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1:>2}/{EPOCHS}", leave=False):
                imgs, masks = imgs.to(device), masks.to(device)

                optimizer.zero_grad()
                outputs = model(imgs)
                loss    = loss_fn(outputs, masks)
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            avg = total_loss / len(train_loader)
            history.append(avg)

            # Save best model
            if avg < best_loss:
                best_loss = avg
                best_epoch = epoch + 1
                torch.save(model.state_dict(), 'best_model.pth')
                print(f"Epoch {epoch+1:>2}/{EPOCHS}  |  avg loss: {avg:.4f}  ⭐ BEST")
            else:
                print(f"Epoch {epoch+1:>2}/{EPOCHS}  |  avg loss: {avg:.4f}")

        print("\nTraining complete (on CPU)  ✅")
        print(f"Best epoch loss : {best_loss:.4f}  (epoch {best_epoch})")
        print("Best model saved as 'best_model.pth'")
    else:
        raise

Epoch  1/10  |  avg loss: 1.1683  ⭐ BEST


Epoch  2/10  |  avg loss: 0.8660  ⭐ BEST


Epoch  3/10  |  avg loss: 0.7597  ⭐ BEST


Epoch  4/10  |  avg loss: 0.6870  ⭐ BEST


Epoch  5/10  |  avg loss: 0.6497  ⭐ BEST


Epoch  6/10  |  avg loss: 0.5913  ⭐ BEST


Epoch  7/10  |  avg loss: 0.5489  ⭐ BEST


Epoch  8/10  |  avg loss: 0.5290  ⭐ BEST


Epoch  9/10  |  avg loss: 0.5269  ⭐ BEST


Epoch 10/10  |  avg loss: 0.4983  ⭐ BEST

Training complete  ✅
Best epoch loss : 0.4983  (epoch 10)
Best model saved as 'best_model.pth'


---
## 🔢 Cell 10 — RLE Encoder

**Run-Length Encoding (RLE)** is a compact way to represent binary masks. Instead of storing every pixel value, it stores alternating *(start_position, run_length)* pairs for runs of foreground (`1`) pixels.

### Example
```
Binary mask (flattened, column-major):   0 0 1 1 1 0 1 0
RLE string:                              3 3 7 1
                                         ↑ ↑ ↑ ↑
                                  start=3 len=3  start=7 len=1
```

### Implementation notes
- Flattening uses **Fortran (column-major) order** (`order='F'`) — this matches the RLE convention used by most Kaggle segmentation competitions.
- Sentinel zeros are prepended and appended before diff-detection so boundary runs are captured correctly.
- If the mask is entirely zero (no flood pixels), the function safely returns `"0 0"`.

A small unit-test is run at the bottom to confirm the function behaves correctly.

In [10]:
def rle_encode(mask: np.ndarray) -> str:
    """
    Encode a binary 2-D mask as a run-length encoded string.
    Uses Fortran (column-major) pixel ordering.

    Parameters
    ----------
    mask : np.ndarray  shape (H, W), dtype uint8, values in {0, 1}

    Returns
    -------
    str  RLE string, e.g. '3 3 7 1'
    """
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(map(str, runs)) if len(runs) else "0 0"

# ── Unit test ────────────────────────────────────────────────────────────────
test_mask = np.array([[0, 1, 1], [0, 1, 0]], dtype=np.uint8)
print("Test mask:")
print(test_mask)
print("RLE output:", rle_encode(test_mask))
print("Empty mask:", rle_encode(np.zeros((4, 4), dtype=np.uint8)))

Test mask:
[[0 1 1]
 [0 1 0]]
RLE output: 3 3
Empty mask: 0 0


---
## 🔍 Cell 11 — Inference

### Steps

| Step | What happens |
|---|---|
| `model.eval()` | Switches BatchNorm & Dropout to inference mode |
| `torch.no_grad()` | Disables gradient tracking → saves memory & speeds up inference |
| `torch.argmax(dim=1)` | Picks the class with the highest logit per pixel → `(B, H, W)` class index map |
| `preds == FLOOD_CLASS` | Binarises to the flood class (index 1) → `uint8` binary mask |
| `rle_encode(...)` | Compresses the binary mask for the submission CSV |

> ⚠️ **Class index assumption:** `FLOOD_CLASS = 1` means class index 1 represents flooded pixels. Verify this against the competition's label documentation before submitting. Change the constant above if needed.

In [11]:
FLOOD_CLASS = 1  # class index representing flooded pixels

model.eval()
results = []

with torch.no_grad():
    for imgs, ids in tqdm(test_loader, desc="Inference"):
        imgs  = imgs.to(device)
        preds = model(imgs)                               # (B, 3, H, W) logits
        preds = torch.argmax(preds, dim=1).cpu().numpy() # (B, H, W) class map

        for i in range(len(ids)):
            flood_mask = (preds[i] == FLOOD_CLASS).astype(np.uint8)
            rle        = rle_encode(flood_mask)
            results.append([ids[i], rle])

print(f"\nInference complete — {len(results):,} predictions ✅")

# Quick validation: show prediction stats
flood_pixels = sum(len(rle.split()) // 2 for _, rle in results if rle != "0 0")
total_pixels = len(results) * 512 * 512
flood_percentage = (flood_pixels / total_pixels) * 100
print(".2f")

Inference: 100%|██████████| 19/19 [00:02<00:00,  7.11it/s]


Inference complete — 19 predictions ✅
.2f


---
## 📤 Cell 12 — Build & Save Submission

The submission CSV must have exactly two columns:

| Column | Description |
|---|---|
| `id` | Sample ID — matches the test filename without the `_image.tif` suffix |
| `rle_mask` | RLE-encoded binary flood mask for that sample |

Rows are sorted by `id` to match the expected submission order. A summary is printed showing the total row count and how many samples have an empty mask (all-zero prediction), which is a useful sanity check.

After this cell runs, `submission.csv` will appear in `/kaggle/working/`. Use the **Submit** button in the top-right corner of the notebook to upload it to the leaderboard.

> 📌 **Always verify** the row count equals the number of test images before submitting.

In [12]:
df = pd.DataFrame(results, columns=["id", "rle_mask"])
df = df.sort_values("id").reset_index(drop=True)

empty_masks = (df["rle_mask"] == "0 0").sum()
print(f"Total rows   : {len(df):,}")
print(f"Empty masks  : {empty_masks:,}  ({100*empty_masks/len(df):.1f}%)")
print()
print(df.head(5).to_string(index=False))

df.to_csv("submission.csv", index=False)
print("\n✅ submission.csv saved to /kaggle/working/")

Total rows   : 19
Empty masks  : 0  (0.0%)

                          id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                